# GST on NTU RGB+D 60 (Kaggle)
This notebook clones the public project repository; enable *Internet* in the notebook settings.

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

REPO = "Muhammad-Huzifa/gst-ntu-skeleton-action-recognition"
WORKING = Path("/kaggle/working")
REPO_DIR = WORKING / "gst_repo"
os.chdir(WORKING)
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(["git", "clone", "--quiet", f"https://github.com/{REPO}.git", str(REPO_DIR)], cwd=WORKING, check=True)
subprocess.run(["git", "log", "-3", "--oneline"], cwd=REPO_DIR, check=True)
os.chdir(REPO_DIR)
print(f"Repository ready at {Path.cwd()}")

## 1. Preprocess (3D, 2 persons, spine-centred + view-aligned) → one packed array
Re-run only if `/kaggle/working/ntu60_packed` is missing. Save it as a Kaggle Dataset afterwards to skip this step next time (~3.4 GB).

In [ ]:
!python -m gst.data.ntu_preprocess --input /kaggle/input/datasets/huzifaulhaq/ntu-60/nturgb+d_skeletons --out /kaggle/working/ntu60_packed --workers 4

## 2. Quick sanity run (2 epochs on 2000 samples) before the long run

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/smoke --epochs 2 --max-train 2000 --eval-every 1

## 3. Train GST (cross-subject). Re-run this cell with `--resume` if the session dies; checkpoints are in /kaggle/working/outputs/gst

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/gst --arch gst --epochs 120 --resume

## 4. Baseline on the SAME pipeline (your old architecture), needed to attribute any gain to the architecture rather than the data fixes

In [ ]:
!python train_ntu.py --data /kaggle/working/ntu60_packed --out /kaggle/working/outputs/baseline --arch baseline --epochs 120 --resume

In [ ]:
import json
for n in ["gst", "baseline"]:
    result_path = Path(f"/kaggle/working/outputs/{n}/final.json")
    if result_path.is_file():
        print(n, json.loads(result_path.read_text())["final"])
    else:
        print(n, "not finished")

## 5. Publish completed runs to GitHub Releases (optional)
Before running this cell, enable Internet and add a Kaggle Secret named `GITHUB_TOKEN` with fine-grained access to this repository and **Contents: Read and write** permission. Releases and their assets are public because this repository is public. Each completed GST or baseline run gets its own unique release containing final weights, metrics, training history, and run metadata. Resume checkpoints and the NTU dataset are not uploaded.

In [ ]:
from kaggle_secrets import UserSecretsClient
from gst.publish import publish_run

OUTPUTS = WORKING / "outputs"
completed_runs = [
    (name, OUTPUTS / name)
    for name in ("gst", "baseline")
    if (OUTPUTS / name / "final.json").is_file()
    and (OUTPUTS / name / "final_weights.pth").is_file()
    and not (OUTPUTS / name / "github_release.json").is_file()
]
if not completed_runs:
    print("No completed GST or baseline runs to publish.")
else:
    github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    try:
        source_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
        ).strip()
        for run_name, run_dir in completed_runs:
            published = publish_run(
                run_dir=run_dir,
                repo=REPO,
                token=github_token,
                run_name=run_name,
                source_commit=source_commit,
            )
            (run_dir / "github_release.json").write_text(json.dumps(published, indent=2))
            print(
                f"Published {run_name}: {published['html_url']} "
                f"(tag {published['tag']}, asset {published['asset_name']}, "
                f"{published['asset_size']:,} bytes)"
            )
    finally:
        del github_token